## 1. Imports & Configuration

In [1]:
import os
import json
import random
import re
import torch
from pathlib import Path
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
from convokit import Corpus, download

/home/ubuntu/persona_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────────

SIM_MODEL_ID   = "Qwen/Qwen3-4B-Thinking-2507" #"Qwen/Qwen3.5-9B-Base" 
JUDGE_MODEL_ID = "Qwen/Qwen2.5-14B-Instruct"

HF_TOKEN=""

# How many conversations to run
N_CONVERSATIONS = 10
RANDOM_SEED     = 42

# Generation hyper-parameters for the base (simulation) model
SIM_MAX_NEW_TOKENS = 200
SIM_TEMPERATURE    = 0.7
SIM_TOP_P          = 0.9

# Generation hyper-parameters for the judge
JUDGE_MAX_NEW_TOKENS = 1024
JUDGE_TEMPERATURE    = 0.3

# Output directories
BASE_DIR    = Path("../data/persuasion_simulation_think")
SIM_DIR     = BASE_DIR / "simulations"
JUDGE_DIR   = BASE_DIR / "judge_annotations"
SIM_DIR.mkdir(parents=True, exist_ok=True)
JUDGE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Simulation output : {SIM_DIR}")
print(f"Judge output      : {JUDGE_DIR}")

Simulation output : ../data/persuasion_simulation_think/simulations
Judge output      : ../data/persuasion_simulation_think/judge_annotations


## 2. Load Simulation Model (Base — No Chat Template)

In [3]:
print(f"Loading simulation model: {SIM_MODEL_ID}")

sim_tokenizer = AutoTokenizer.from_pretrained(SIM_MODEL_ID)
sim_model     = AutoModelForCausalLM.from_pretrained(
    SIM_MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    token=HF_TOKEN
)
sim_model.eval()

# Ensure there is a pad token
if sim_tokenizer.pad_token is None:
    sim_tokenizer.pad_token = sim_tokenizer.eos_token

print("Simulation model loaded.")

Loading simulation model: Qwen/Qwen3-4B-Thinking-2507


`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 398/398 [00:01<00:00, 383.52it/s]


Simulation model loaded.


## 3. Load Dataset

In [4]:
print("Downloading / loading Persuasion for Good corpus...")
corpus = Corpus(filename=download("persuasionforgood-corpus"))

all_convos = list(corpus.iter_conversations())
print(f"Total conversations in corpus: {len(all_convos)}")

# Quick peek at speaker metadata keys
sample_spk = next(corpus.iter_speakers())
print("\nSample speaker metadata keys:", list(sample_spk.meta.keys()))

Dataset already exists at /home/ubuntu/.convokit/saved-corpora/persuasionforgood-corpus
Total conversations in corpus: 1017

Sample speaker metadata keys: ['extrovert', 'agreeable', 'conscientious', 'neurotic', 'open', 'care', 'fairness', 'loyalty', 'authority', 'purity', 'freedom', 'conform', 'tradition', 'benevolence', 'universalism', 'self_direction', 'stimulation', 'hedonism', 'achievement', 'power', 'security', 'rational', 'intuitive', 'age', 'sex', 'race', 'edu', 'marital', 'employment', 'income', 'religion', 'ideology']


## 4. Select 50 Conversations with Complete Persuadee Persona Data

In [5]:
def get_speaker_role(utterance):
    """Return 'Persuader' (role=0) or 'Persuadee' (role=1) from utterance meta."""
    role = utterance.meta.get("role")
    if role == 0:
        return "Persuader"
    elif role == 1:
        return "Persuadee"
    return ""


def _utt_sort_key(utt):
    """Sort by (user_turn_id, role) so within each turn persuader (0) comes first."""
    turn_id = utt.meta.get("user_turn_id", 0)
    role    = utt.meta.get("role", 0)
    return (turn_id, role)


def get_utterances_in_order(conversation):
    """Return utterances sorted by (user_turn_id, role)."""
    utts = list(conversation.iter_utterances())
    return sorted(utts, key=_utt_sort_key)


def get_ordered_turns(conversation):
    """
    Group utterances into (persuader_message, human_response) pairs.
    Consecutive same-role utterances are concatenated into one block.
    """
    utts = get_utterances_in_order(conversation)
    turns = []
    i = 0
    while i < len(utts):
        role = get_speaker_role(utts[i])
        if role == "Persuader":
            persuader_parts = []
            while i < len(utts) and get_speaker_role(utts[i]) == "Persuader":
                persuader_parts.append(utts[i].text.strip())
                i += 1
            persuader_msg = " ".join(persuader_parts)

            persuadee_parts = []
            while i < len(utts) and get_speaker_role(utts[i]) == "Persuadee":
                persuadee_parts.append(utts[i].text.strip())
                i += 1
            human_response = " ".join(persuadee_parts)

            if persuader_msg and human_response:
                turns.append({
                    "persuader_message": persuader_msg,
                    "human_response": human_response,
                })
        else:
            i += 1
    return turns


def get_persuadee(conversation):
    """Return the Speaker object for the persuadee in the conversation."""
    for utt in get_utterances_in_order(conversation):
        if get_speaker_role(utt) == "Persuadee":
            return utt.speaker
    return None


def has_persona_data(speaker):
    """Check whether the speaker has at least one demographic or personality field."""
    if speaker is None:
        return False
    meta = speaker.meta
    return any(meta.get(k) not in (None, "", "N/A") for k in ["age", "sex", "race", "edu", "extrovert"])

In [6]:
random.seed(RANDOM_SEED)

# Filter: conversations with at least 2 turns and persuadee persona data
valid_convos = []
for conv in all_convos:
    turns = get_ordered_turns(conv)
    persuadee = get_persuadee(conv)
    if len(turns) >= 2 and has_persona_data(persuadee):
        valid_convos.append(conv)

print(f"Valid conversations (≥2 turns + persona data): {len(valid_convos)}")

selected_convos = random.sample(valid_convos, min(N_CONVERSATIONS, len(valid_convos)))
print(f"Selected {len(selected_convos)} conversations for simulation.")

Valid conversations (≥2 turns + persona data): 1017
Selected 10 conversations for simulation.


## 5. Persona Extraction

In [16]:
# ── Exact metadata keys confirmed from corpus inspection ──────────────────────

DEMO_KEYS = {
    "age": "age", "sex": "sex", "race": "race", "edu": "education",
    "marital": "marital_status", "employment": "employment",
    "income": "income", "religion": "religion", "ideology": "ideology",
}

BIG5_KEYS = {
    "extrovert": "extroversion", "agreeable": "agreeableness",
    "conscientious": "conscientiousness", "neurotic": "neuroticism", "open": "openness",
}

MF_KEYS = {
    "care": "care", "fairness": "fairness", "loyalty": "loyalty",
    "authority": "authority", "purity": "purity", "freedom": "freedom",
}

SCHWARTZ_KEYS = {
    "conform": "conformity", "tradition": "tradition", "benevolence": "benevolence",
    "universalism": "universalism", "self_direction": "self_direction",
    "stimulation": "stimulation", "hedonism": "hedonism", "achievement": "achievement",
    "power": "power", "security": "security",
}

DM_KEYS = {
    "rational": "rational", "intuitive": "intuitive",
}


def _extract_group(meta, key_map):
    """Extract available fields using {raw_key: label} map; skip missing/null values."""
    return {
        label: meta[raw_key]
        for raw_key, label in key_map.items()
        if meta.get(raw_key) not in (None, "", "N/A")
    }


def extract_persona(speaker):
    """Return a fully structured persona dict from a ConvoKit Speaker."""
    meta = speaker.meta
    return {
        "speaker_id"       : speaker.id,
        "demographics"     : _extract_group(meta, DEMO_KEYS),
        "big_five"         : _extract_group(meta, BIG5_KEYS),
        "moral_foundations": _extract_group(meta, MF_KEYS),
        "schwartz_values"  : _extract_group(meta, SCHWARTZ_KEYS),
        "decision_making"  : _extract_group(meta, DM_KEYS),
    }

## 6. Prompt Building (Base Model — No Chat Template)

In [17]:
def persona_to_text(persona):
    """Convert structured persona dict to a readable natural-language description."""
    lines = []

    d = persona.get("demographics", {})
    if d:
        demo_str = ", ".join(f"{k}: {v}" for k, v in d.items())
        lines.append(f"Demographics — {demo_str}.")

    b5 = persona.get("big_five", {})
    if b5:
        b5_str = ", ".join(f"{k} = {v}" for k, v in b5.items())
        lines.append(f"Big Five personality (scale 1–5) — {b5_str}.")

    mf = persona.get("moral_foundations", {})
    if mf:
        mf_str = ", ".join(f"{k} = {v}" for k, v in mf.items())
        lines.append(f"Moral foundations (scale 1–6) — {mf_str}.")

    sv = persona.get("schwartz_values", {})
    if sv:
        sv_str = ", ".join(f"{k} = {v}" for k, v in sv.items())
        lines.append(f"Schwartz values (scale 1–6) — {sv_str}.")

    dm = persona.get("decision_making", {})
    if dm:
        dm_str = ", ".join(f"{k} = {v}" for k, v in dm.items())
        lines.append(f"Decision-making style (scale 1–5) — {dm_str}.")

    return "\n".join(lines) if lines else "No persona data available."


def build_sim_prompt(persona, history, persuader_message):
    """
    Build a plain-text completion prompt for the base model.
    The model is shown the persona and the conversation so far, then prompted
    to continue as the persuadee.

    `history` is a list of dicts: [{"persuader": ..., "you": ...}]
    The model will complete the final "You:" turn.
    """
    persona_text = persona_to_text(persona)

    prompt = (
        "The following is a real-time conversation between a Persuader and You.\n"
        "The Persuader is trying to convince You to donate to a charity called \'Save the Children\'.\n\n"
        "Your personal background:\n"
        f"{persona_text}\n\n"
        "Instructions: Respond naturally as yourself based on your background above. "
        "Your response should feel authentic to who you are — "
        "your values, personality, and worldview. "
        "Keep your reply conversational (1–4 sentences).\n\n"
        "Conversation:\n"
    )

    # Append conversation history (using real human responses to keep context faithful)
    for turn in history:
        prompt += f"Persuader: {turn['persuader']}\n"
        prompt += f"You: {turn['you']}\n"

    # Append the current persuader message and leave the completion blank
    prompt += f"Persuader: {persuader_message}\n"
    prompt += "You:"

    return prompt

## 7. Generation Function (Standard Completion)

In [18]:
def parse_model_output(raw_text, stop_strings=()):
    """
    Extract <think>...</think> reasoning and the response that follows it.
    If no <think> block is present, reasoning is empty and the full text is the response.
    Stop strings are applied to the response portion only.

    Returns: {"reasoning": str, "response": str}
    """
    reasoning = ""
    response  = raw_text

    think_match = re.search(r"<think>(.*?)</think>", raw_text, re.DOTALL)
    if think_match:
        reasoning = think_match.group(1).strip()
        response  = raw_text[think_match.end():].strip()

    for stop in stop_strings:
        if stop in response:
            response = response[:response.index(stop)].strip()

    return {"reasoning": reasoning, "response": response}


def generate_completion(prompt, model, tokenizer,
                        max_new_tokens=SIM_MAX_NEW_TOKENS,
                        temperature=SIM_TEMPERATURE,
                        top_p=SIM_TOP_P,
                        stop_strings=("\nPersuader:", "\nYou:")):
    """
    Run standard text completion on `prompt` using the base model.
    No chat template is applied.
    Returns a dict: {"reasoning": <think> block content, "response": final response text}
    """
    inputs    = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048)
    inputs    = {k: v.to(model.device) for k, v in inputs.items()}
    input_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    raw_text = tokenizer.decode(
        output_ids[0][input_len:],
        skip_special_tokens=True
    ).strip()

    return parse_model_output(raw_text, stop_strings=stop_strings)

## 8. Simulate a Single Conversation

In [19]:
def simulate_conversation(conversation, model, tokenizer):
    """
    Run turn-by-turn simulation for one conversation.
    Returns a result dict ready to be saved as JSON.

    History is built using the REAL human responses so that the persuader's
    subsequent messages remain contextually grounded.
    """
    persuadee = get_persuadee(conversation)
    persona   = extract_persona(persuadee)
    turns     = get_ordered_turns(conversation)

    history = []   # list of {persuader, you} — populated with real human replies
    result_turns = []

    for turn_idx, turn in enumerate(turns):
        persuader_msg  = turn["persuader_message"]
        human_response = turn["human_response"]

        prompt      = build_sim_prompt(persona, history, persuader_msg)
        llm_response = generate_completion(prompt, model, tokenizer)

        result_turns.append({
            "turn_index"       : turn_idx,
            "persuader_message": persuader_msg,
            "human_response"   : human_response,
            "llm_response"     : llm_response,
        })

        # Advance history using the REAL human response
        history.append({"persuader": persuader_msg, "you": human_response})

    return {
        "conversation_id"  : conversation.id,
        "persuadee_id"     : persuadee.id,
        "persuadee_persona": persona,
        "turns"            : result_turns,
    }

## 9. Run Simulation on 50 Conversations & Save JSONs

In [20]:
def save_simulation(result, output_dir):
    conv_id  = result["conversation_id"]
    out_path = output_dir / f"conv_{conv_id}.json"
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(result, f, indent=2, ensure_ascii=False)
    return out_path


failed = []

for conv in tqdm(selected_convos, desc="Simulating conversations"):
    try:
        result   = simulate_conversation(conv, sim_model, sim_tokenizer)
        out_path = save_simulation(result, SIM_DIR)
        print(f"  Saved: {out_path.name}  ({len(result['turns'])} turns)")
    except Exception as e:
        print(f"  ERROR on {conv.id}: {e}")
        failed.append(conv.id)

print(f"\nDone. {len(selected_convos) - len(failed)}/{len(selected_convos)} conversations saved.")
if failed:
    print(f"Failed: {failed}")

Simulating conversations:  10%|█         | 1/10 [01:41<15:17, 101.99s/it]

  Saved: conv_13457.json  (10 turns)


Simulating conversations:  10%|█         | 1/10 [03:16<29:26, 196.30s/it]


KeyboardInterrupt: 

---
## 10. Judge — Load Qwen2.5-14B-Instruct

In [ ]:
# Free simulation model memory before loading the judge (optional but recommended
# if running on a single GPU)
# del sim_model
# torch.cuda.empty_cache()

print(f"Loading judge model: {JUDGE_MODEL_ID}")

judge_tokenizer = AutoTokenizer.from_pretrained(JUDGE_MODEL_ID)
judge_model     = AutoModelForCausalLM.from_pretrained(
    JUDGE_MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
)
judge_model.eval()

print("Judge model loaded.")

## 11. Judge Prompt & Generation

In [ ]:
JUDGE_SYSTEM_PROMPT = """\
You are an expert conversation analyst specialising in persuasion dynamics and belief change.
You will be given a conversation between a Persuader and a Persuadee, displayed turn by turn.
For each turn you are shown both the REAL human response and an LLM-simulated response.

Your task is to produce a structured JSON analysis with the following schema:

{
  "conversation_id": "<id>",
  "turn_annotations": [
    {
      "turn_index": <int>,
      "mind_change_human": <null | {"signal": "<brief label>", "evidence": "<quoted text or explanation>"}>,
      "mind_change_llm":   <null | {"signal": "<brief label>", "evidence": "<quoted text or explanation>"}>,
      "alignment": "<describe key points where human and LLM responses agree or are similar>",
      "distinctions": "<describe key differences in tone, stance, content, or emotional register>"
    }
  ],
  "mind_change_summary": {
    "human": [<list of turn_index values where mind-change was detected in human>],
    "llm":   [<list of turn_index values where mind-change was detected in LLM>]
  }
}

Guidelines:
- "mind_change" includes: explicit agreement to donate, softening of resistance, 
  hedging language ("maybe", "I hadn't thought of that"), increased engagement, 
  or any shift from scepticism toward openness.
- Set mind_change to null if there is no signal in that turn.
- Be concise but specific. Quote short phrases as evidence where possible.
- Return ONLY the JSON object. No extra text before or after."""


def build_judge_user_message(simulation_result):
    """Format the simulation result as a readable conversation block for the judge."""
    conv_id = simulation_result["conversation_id"]
    lines   = [f"Conversation ID: {conv_id}\n"]

    for t in simulation_result["turns"]:
        lines.append(f"--- Turn {t['turn_index']} ---")
        lines.append(f"Persuader   : {t['persuader_message']}")
        lines.append(f"Human       : {t['human_response']}")
        lines.append(f"LLM (simul.): {t['llm_response']}")
        lines.append("")

    return "\n".join(lines)


def extract_json_from_text(text):
    """Extract the first JSON object from model output, handling markdown fences."""
    # Try to strip ```json ... ``` fences first
    fenced = re.search(r"```(?:json)?\s*({.*?})\s*```", text, re.DOTALL)
    if fenced:
        return json.loads(fenced.group(1))
    # Fall back to finding the outermost { ... }
    start = text.find("{")
    end   = text.rfind("}")
    if start != -1 and end != -1:
        return json.loads(text[start:end + 1])
    raise ValueError("No JSON object found in judge output.")


def run_judge(simulation_result, model, tokenizer):
    """
    Run the judge model on a single simulation result.
    Returns a parsed JSON annotation dict.
    """
    user_message = build_judge_user_message(simulation_result)

    messages = [
        {"role": "system", "content": JUDGE_SYSTEM_PROMPT},
        {"role": "user",   "content": user_message},
    ]

    # Apply the instruct chat template
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs   = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=4096)
    inputs   = {k: v.to(model.device) for k, v in inputs.items()}
    in_len   = inputs["input_ids"].shape[1]

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=JUDGE_MAX_NEW_TOKENS,
            temperature=JUDGE_TEMPERATURE,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    raw_output = tokenizer.decode(
        output_ids[0][in_len:],
        skip_special_tokens=True,
    ).strip()

    try:
        annotation = extract_json_from_text(raw_output)
    except (json.JSONDecodeError, ValueError):
        # Save raw output so nothing is lost
        annotation = {"conversation_id": simulation_result["conversation_id"],
                      "raw_output": raw_output,
                      "parse_error": True}

    return annotation

## 12. Run Judge on All Saved Simulation Files

In [ ]:
sim_files = sorted(SIM_DIR.glob("conv_*.json"))
print(f"Found {len(sim_files)} simulation files to judge.\n")

judge_failed = []

for sim_path in tqdm(sim_files, desc="Running judge"):
    with open(sim_path, "r", encoding="utf-8") as f:
        sim_result = json.load(f)

    conv_id = sim_result["conversation_id"]

    try:
        annotation = run_judge(sim_result, judge_model, judge_tokenizer)

        judge_path = JUDGE_DIR / f"judge_conv_{conv_id}.json"
        with open(judge_path, "w", encoding="utf-8") as f:
            json.dump(annotation, f, indent=2, ensure_ascii=False)

        parse_ok = not annotation.get("parse_error", False)
        print(f"  Saved: {judge_path.name}  (parse_ok={parse_ok})")

    except Exception as e:
        print(f"  ERROR judging {conv_id}: {e}")
        judge_failed.append(conv_id)

print(f"\nJudge done. {len(sim_files) - len(judge_failed)}/{len(sim_files)} files annotated.")
if judge_failed:
    print(f"Failed: {judge_failed}")

## 13. Quick Sanity Check — Preview One Result

In [ ]:
# Preview the first simulation + its judge annotation side-by-side
preview_sim_path   = sorted(SIM_DIR.glob("conv_*.json"))[0]
conv_id_preview    = preview_sim_path.stem.replace("conv_", "")
preview_judge_path = JUDGE_DIR / f"judge_conv_{conv_id_preview}.json"

with open(preview_sim_path, "r") as f:
    sim_preview = json.load(f)

print(f"=== Conversation {sim_preview['conversation_id']} ===")
print(f"Persuadee: {sim_preview['persuadee_id']}")
print()

for turn in sim_preview["turns"]:
    print(f"[Turn {turn['turn_index']}]")
    print(f"  Persuader : {turn['persuader_message'][:120]}")
    print(f"  Human     : {turn['human_response'][:120]}")
    print(f"  LLM       : {turn['llm_response'][:120]}")
    print()

if preview_judge_path.exists():
    print("=== Judge Annotations ===")
    with open(preview_judge_path, "r") as f:
        judge_preview = json.load(f)
    print(json.dumps(judge_preview, indent=2))
else:
    print("Judge annotation not yet available for this conversation.")